# SQL Query Agent in LangChain

Langchain provides a SQL Agent which offers a flexible way of interacting with SQL Databases. The SQL Agent system can answer questions intelligently by utilizing both the schema and content of the database, allowing it to provide insights into specific tables or records as needed. It can effectively handle and recover from errors by executing a generated query, catching any errors in the traceback, and adjusting the query accordingly. The system is optimized to query the database as many times as necessary to gather the information required to answer user questions fully. Additionally, it conserves resources by only retrieving the schema of relevant tables, ensuring efficient use of tokens and improving performance.


## Lab Description:

This lab explores how to use LangChain’s SQL Query Agent to interact with a SQLite database (chinook.db). Participants will learn two methods to fetch table names and retrieve table information before using the SQL Agent to execute queries. The lab demonstrates how the agent can automatically generate SQL queries, retrieve data, and provide table descriptions, making database interactions more efficient and intuitive.

## Lab Objectives

- Learn how to connect LangChain to a SQLite database (chinook.db).
- Explore methods to fetch table names and retrieve table information.
- Utilize LangChain’s SQL Query Agent to execute queries dynamically.
- Generate and interpret table descriptions using the SQL Agent.

## Step 0 — Build the Chinook database (run this first)

This lab queries a SQLite file called **`Chinook.db`**. The repository ships the database
itself, but it can always be rebuilt from `Chinook_Sqlite.sql` — and the cell below does
that automatically if the file is missing or incomplete.

> **Note on `sqlite3`.** The lab image does not include the `sqlite3` *command-line tool*,
> so `sqlite3 Chinook.db < Chinook_Sqlite.sql` fails with `command not found`. Python's
> standard-library `sqlite3` *module* is always available, so the cell below needs no
> installs, no network and no elevated rights.

The cell is idempotent: run it as often as you like, it only rebuilds when it has to.


In [1]:
# [FDP-PATCHED] Step 0 - ensure Chinook.db exists, without needing the sqlite3 CLI.
# Idempotent: rebuilds only if the database is missing, corrupt or incomplete.
from pathlib import Path
import sqlite3

DB_PATH = Path("Chinook.db")
SQL_PATH = Path("Chinook_Sqlite.sql")
EXPECTED_ALBUMS = 347


def chinook_is_complete(db_path=DB_PATH):
    """True only if the file opens and holds the full Chinook dataset."""
    if not db_path.exists():
        return False
    try:
        con = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True)
        try:
            return con.execute("SELECT COUNT(*) FROM Album").fetchone()[0] == EXPECTED_ALBUMS
        finally:
            con.close()
    except sqlite3.Error:
        return False


def build_chinook(db_path=DB_PATH, sql_path=SQL_PATH):
    if not sql_path.exists():
        raise FileNotFoundError(
            f"{sql_path} not found. Start Jupyter in the Lab14-SQL-Agent folder, "
            "or run: %cd ~/fdp/labs/Lab14-SQL-Agent"
        )
    print(f"Building {db_path} from {sql_path} ...")
    # utf-8 keeps accented names intact (e.g. 'Cassia Eller'); latin-1 is a lossless fallback.
    raw = sql_path.read_bytes()
    try:
        sql = raw.decode("utf-8-sig")
    except UnicodeDecodeError:
        sql = raw.decode("latin-1")
    db_path.unlink(missing_ok=True)
    con = sqlite3.connect(db_path)
    try:
        con.executescript(sql)
        con.commit()
    finally:
        con.close()


if chinook_is_complete():
    print(f"Chinook.db is present and complete ({EXPECTED_ALBUMS} albums).")
else:
    build_chinook()
    assert chinook_is_complete(), "Build finished but the database is still incomplete."
    print(f"Chinook.db built and verified ({EXPECTED_ALBUMS} albums).")

# Quick look at what we now have to work with.
with sqlite3.connect(DB_PATH) as con:
    tables = [r[0] for r in con.execute(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")]
    print(f"{len(tables)} tables:", ", ".join(tables))


Chinook.db is present and complete (347 albums).
11 tables: Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, Track


## Load the LLM 

First step is to load the LLM. We use gemma2 9 billion parameter model. 

In [2]:
from langchain_ollama.llms import OllamaLLM

In [3]:
llm = OllamaLLM(model = "gemma2:9b", base_url="http://10.79.253.112:11434", temperature=0)

## The Langchain SQL Database Toolkit

The `SQLDatabaseToolkit` contains tools that helps interact with a `SQL` database. `SQLDatabaseToolkit` enables SQL Agents to answer questions using data in a database. `.from_uri` method constructs a SQLAlchemy engine from URI. The URI string specifies the path to the database. The database should be located in the same directory as the notebook. We use SQLLite as it is lightweight and serverless. Working with agents are a lot easier when using lightweight SQL Engines. 

In [4]:
from langchain_community.utilities import SQLDatabase

In [5]:
db = SQLDatabase.from_uri("sqlite:///Chinook.db")

In [6]:
# [FDP-PATCHED] Fail fast if the notebook is running outside the lab folder:
# SQLAlchemy happily opens a brand-new empty file for a missing sqlite path.
assert "Album" in db.get_usable_table_names(), (
    "Chinook.db has no Album table - re-run the Step 0 cell from the "
    "Lab14-SQL-Agent folder."
)
print(db.run("SELECT COUNT(*) AS albums FROM Album;"))


[(347,)]


## `get_usable_table_names` Method

The `get_usable_table_names` returns the names of all tables in the database.

In [7]:
db.get_usable_table_names()

['Album',
 'Artist',
 'Customer',
 'Employee',
 'Genre',
 'Invoice',
 'InvoiceLine',
 'MediaType',
 'Playlist',
 'PlaylistTrack',
 'Track']

## `get_table_info` Method

`get_table_info` method can be called to fetch details about specific tables. 

In [8]:
print(db.get_table_info(["Album"]))


CREATE TABLE "Album" (
	"AlbumId" INTEGER NOT NULL, 
	"Title" NVARCHAR(160) NOT NULL, 
	"ArtistId" INTEGER NOT NULL, 
	PRIMARY KEY ("AlbumId"), 
	FOREIGN KEY("ArtistId") REFERENCES "Artist" ("ArtistId")
)

/*
3 rows from Album table:
AlbumId	Title	ArtistId
1	For Those About To Rock We Salute You	1
2	Balls to the Wall	2
3	Restless and Wild	2
*/


This returned a description about the Albums table. `get_table_info` accepts a list of strings so that description about multiple tables can be fetched. 

In [9]:
print(db.get_table_info(["Album", "Artist"]))


CREATE TABLE "Album" (
	"AlbumId" INTEGER NOT NULL, 
	"Title" NVARCHAR(160) NOT NULL, 
	"ArtistId" INTEGER NOT NULL, 
	PRIMARY KEY ("AlbumId"), 
	FOREIGN KEY("ArtistId") REFERENCES "Artist" ("ArtistId")
)

/*
3 rows from Album table:
AlbumId	Title	ArtistId
1	For Those About To Rock We Salute You	1
2	Balls to the Wall	2
3	Restless and Wild	2
*/


CREATE TABLE "Artist" (
	"ArtistId" INTEGER NOT NULL, 
	"Name" NVARCHAR(120), 
	PRIMARY KEY ("ArtistId")
)

/*
3 rows from Artist table:
ArtistId	Name
1	AC/DC
2	Accept
3	Aerosmith
*/


## The `create_sql_agent` Method

`create_sql_agent` is used to create a SQLAgent from an LLM and database. We provide the LLM, and the db object we initialized to the `create_sql_agent`method. 

In [10]:
from langchain_community.agent_toolkits import create_sql_agent
from langchain.agents import AgentType

In [11]:
agent = create_sql_agent(llm, db=db)

Now that we have initialized the agent, we can invoke it with the `.invoke` method. 

In [12]:
result = agent.invoke("How many different Artists are in the database?")

In [13]:
result

{'input': 'How many different Artists are in the database?',
 'output': 'There are 275 different artists in the database.'}

The agent generated a response by querying the database. 

We can set `verbose=True` to see the step by step thought process of the agent. 

In [14]:
agent = create_sql_agent(llm, db=db, verbose=True)

In [15]:
agent.invoke("List the total sales per country")



> Entering new SQL Agent Executor chain...
Action: sql_db_list_tables
Thought: The 'Invoice' table likely contains sales information and the 'Customer' table likely has country information. I should query the schema of these tables.
Action: sql_db_schema
Action Input: Invoice, Customer
CREATE TABLE "Customer" (
	"CustomerId" INTEGER NOT NULL, 
	"FirstName" NVARCHAR(40) NOT NULL, 
	"LastName" NVARCHAR(20) NOT NULL, 
	"Company" NVARCHAR(80), 
	"Address" NVARCHAR(70), 
	"City" NVARCHAR(40), 
	"State" NVARCHAR(40), 
	"Country" NVARCHAR(40), 
	"PostalCode" NVARCHAR(10), 
	"Phone" NVARCHAR(24), 
	"Fax" NVARCHAR(24), 
	"Email" NVARCHAR(60) NOT NULL, 
	"SupportRepId" INTEGER, 
	PRIMARY KEY ("CustomerId"), 
	FOREIGN KEY("SupportRepId") REFERENCES "Employee" ("EmployeeId")
)

/*
3 rows from Customer table:
CustomerId	FirstName	LastName	Company	Address	City	State	Country	PostalCode	Phone	Fax	Email	SupportRepId
1	Luís	Gonçalves	Embraer - Empresa Brasileira de Aeronáutica S.A.	Av. Brigadeiro Fari

{'input': 'List the total sales per country',
 'output': 'The top 10 countries with the highest total sales are: USA (13.86), United Kingdom (8.91), Netherlands (8.91), Finland (8.91), Czech Republic (8.91), Canada (8.91), Brazil (8.91), Ireland (5.94), Belgium (5.94), and Norway (3.96).'}

The agent identifies the relevant table based on the user query. It then analyses the schema of the identified table to generate a syntactically correct query to generate the final output. 

## Table Descriptions

SQLAgents can also describe the content of a specific table. That is, it can provide all information on what the table is about. 

In [16]:
agent.invoke("What is the 'Employee' table about ?")



> Entering new SQL Agent Executor chain...
Thought: I should look at the tables in the database to see what I can query.  Then I should query the schema of the most relevant tables.
Action: sql_db_list_tables
Thought: The 'Employee' table seems like a good place to start. I should query its schema.aType, Playlist, PlaylistTrack, Track
Action: sql_db_schema
Action Input: Employee
CREATE TABLE "Employee" (
	"EmployeeId" INTEGER NOT NULL, 
	"LastName" NVARCHAR(20) NOT NULL, 
	"FirstName" NVARCHAR(20) NOT NULL, 
	"Title" NVARCHAR(30), 
	"ReportsTo" INTEGER, 
	"BirthDate" DATETIME, 
	"HireDate" DATETIME, 
	"Address" NVARCHAR(70), 
	"City" NVARCHAR(40), 
	"State" NVARCHAR(40), 
	"Country" NVARCHAR(40), 
	"PostalCode" NVARCHAR(10), 
	"Phone" NVARCHAR(24), 
	"Fax" NVARCHAR(24), 
	"Email" NVARCHAR(60), 
	PRIMARY KEY ("EmployeeId"), 
	FOREIGN KEY("ReportsTo") REFERENCES "Employee" ("EmployeeId")
)

/*
3 rows from Employee table:
EmployeeId	LastName	FirstName	Title	ReportsTo	BirthDate	HireDate	

{'input': "What is the 'Employee' table about ?",
 'output': "The 'Employee' table stores information about employees in a company, including their ID, name, title, manager, hire date, contact details, and address."}

The agent provided a description about the Employee table. 

In [17]:
agent.invoke("Generate query for fetching the names of all the employees")



> Entering new SQL Agent Executor chain...
Action: sql_db_list_tables
Thought: The "Employee" table likely contains employee names. I should query its schema.diaType, Playlist, PlaylistTrack, Track
Action: sql_db_schema
Action Input: Employee
CREATE TABLE "Employee" (
	"EmployeeId" INTEGER NOT NULL, 
	"LastName" NVARCHAR(20) NOT NULL, 
	"FirstName" NVARCHAR(20) NOT NULL, 
	"Title" NVARCHAR(30), 
	"ReportsTo" INTEGER, 
	"BirthDate" DATETIME, 
	"HireDate" DATETIME, 
	"Address" NVARCHAR(70), 
	"City" NVARCHAR(40), 
	"State" NVARCHAR(40), 
	"Country" NVARCHAR(40), 
	"PostalCode" NVARCHAR(10), 
	"Phone" NVARCHAR(24), 
	"Fax" NVARCHAR(24), 
	"Email" NVARCHAR(60), 
	PRIMARY KEY ("EmployeeId"), 
	FOREIGN KEY("ReportsTo") REFERENCES "Employee" ("EmployeeId")
)

/*
3 rows from Employee table:
EmployeeId	LastName	FirstName	Title	ReportsTo	BirthDate	HireDate	Address	City	State	Country	PostalCode	Phone	Fax	Email
1	Adams	Andrew	General Manager	None	1962-02-18 00:00:00	2002-08-14 00:00:00	11120 Jas

{'input': 'Generate query for fetching the names of all the employees',
 'output': 'Adams, Andrew; Edwards, Nancy; Peacock, Jane; Park, Margaret; Johnson, Steve; Mitchell, Michael; King, Robert; Callahan, Laura'}

<div style="text-align: left;">
    <img src="logo.png" alt="flow" width="150" height="100">
</div>